<a href="https://colab.research.google.com/github/delsucflorian/Oncolake_TorchProtein/blob/main/notebooks/02_evaluation_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 — Evaluation pipeline

This notebook defines the evaluation protocol shared by all three models
of the project (sequence baseline, OncoLake baseline, TorchProtein embeddings).
The protocol is fixed **before** any model is trained, to prevent p-hacking.

## Pre-declared protocol

- **Split** : family-aware, using MMseqs2 clustering at 40% sequence identity.
  Proteins from the same cluster stay in the same fold.
- **Cross-validation** : 5-fold GroupKFold (respects clusters).
- **Metrics** : F1 macro (primary), balanced accuracy, MCC, AUC-ROC (secondary).
- **Seeds** : 5 seeds per model (42..46). Reported as mean ± std.
- **Statistical comparison** : Wilcoxon signed-rank on paired scores
  (25 scores per model = 5 seeds × 5 folds).

## Output of this notebook

A reusable function `evaluate_model()` that any downstream notebook can call
with any (X, y) matrix to produce comparable results.

In [3]:
import pandas as pd
import json
import numpy as np
import os

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
features_ref = pd.read_parquet('/content/drive/MyDrive/oncolake_torchprotein/data/features_baseline_ref.parquet')
print(features_ref.columns.tolist())
assert features_ref.shape == (404, 28), f"Shape attendu (404, 28), trouvé {features_ref.shape}"

['n_residues_structure', 'plddt_mean', 'pct_low_confidence', 'radius_of_gyration', 'aa_A', 'aa_C', 'aa_D', 'aa_E', 'aa_F', 'aa_G', 'aa_H', 'aa_I', 'aa_K', 'aa_L', 'aa_M', 'aa_N', 'aa_P', 'aa_Q', 'aa_R', 'aa_S', 'aa_T', 'aa_V', 'aa_W', 'aa_Y', 'accession', 'gene', 'label', 'seq_length']


In [6]:

with open('/content/drive/MyDrive/oncolake_torchprotein/data/manifest.json') as f:
    manifest = json.load(f)
    assert (len(manifest)== 418), f"Attendu 418 , trouvé {len(manifest)}"

In [7]:
accessions_404 = features_ref['accession'].tolist()
print(len(accessions_404))
for i in range (6):
  print(accessions_404[i])

404
P00519
P04198
P04201
P08620
P08922
P09769


In [8]:
manifest_404 = [entry for entry in manifest if entry['accession'] in set(accessions_404)]
assert len(manifest_404) == 404, f"Attendu 404 , trouvé {len(manifest_404)}"

In [9]:
# Vérification : combien d'accessions UNIQUES dans manifest_404 ?
n_unique = len(set(entry['accession'] for entry in manifest_404))
print(f"Entrées dans manifest_404 : {len(manifest_404)}")
print(f"Accessions uniques : {n_unique}")

# Vérification : y a-t-il des doublons ?
from collections import Counter
counts = Counter(entry['accession'] for entry in manifest_404)
duplicates = {acc: count for acc, count in counts.items() if count > 1}
print(f"Accessions dupliquées : {len(duplicates)}")
if duplicates:
    print(f"  Exemples : {list(duplicates.items())[:5]}")

assert len({e['accession'] for e in manifest_404}) == 404, "Doublons détectés !"

Entrées dans manifest_404 : 404
Accessions uniques : 404
Accessions dupliquées : 0


In [10]:
!apt-get install -y mmseqs2 -qq

Selecting previously unselected package libgzstream0:amd64.
(Reading database ... 122797 files and directories currently installed.)
Preparing to unpack .../libgzstream0_1.5+git20171107.9a20658-1_amd64.deb ...
Unpacking libgzstream0:amd64 (1.5+git20171107.9a20658-1) ...
Selecting previously unselected package mmseqs2.
Preparing to unpack .../mmseqs2_15-6f452+ds-2_amd64.deb ...
Unpacking mmseqs2 (15-6f452+ds-2) ...
Setting up libgzstream0:amd64 (1.5+git20171107.9a20658-1) ...
Setting up mmseqs2 (15-6f452+ds-2) ...
Processing triggers for man-db (2.12.0-4build2) ...
Processing triggers for libc-bin (2.39-0ubuntu8.8) ...
/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero_v2.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm_debug.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_loader.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libumf.so.1

In [11]:
!mmseqs version

15-6f452+ds-2


In [12]:
fasta_path = '/content/proteins.fasta'
with open(fasta_path, 'w') as f :
  for entry in manifest_404:
        f.write(f">{entry['accession']}\n")
        f.write(f"{entry['sequence']}\n")

In [13]:
!head -6 /content/proteins.fasta

>P00519
MLEICLKLVGCKSKKGLSSSSSCYLEEALQRPVASDFEPQGLSEAARWNSKENLLAGPSENDPNLFVALYDFVASGDNTLSITKGEKLRVLGYNHNGEWCEAQTKNGQGWVPSNYITPVNSLEKHSWYHGPVSRNAAEYLLSSGINGSFLVRESESSPGQRSISLRYEGRVYHYRINTASDGKLYVSSESRFNTLAELVHHHSTVADGLITTLHYPAPKRNKPTVYGVSPNYDKWEMERTDITMKHKLGGGQYGEVYEGVWKKYSLTVAVKTLKEDTMEVEEFLKEAAVMKEIKHPNLVQLLGVCTREPPFYIITEFMTYGNLLDYLRECNRQEVNAVVLLYMATQISSAMEYLEKKNFIHRDLAARNCLVGENHLVKVADFGLSRLMTGDTYTAHAGAKFPIKWTAPESLAYNKFSIKSDVWAFGVLLWEIATYGMSPYPGIDLSQVYELLEKDYRMERPEGCPEKVYELMRACWQWNPSDRPSFAEIHQAFETMFQESSISDEVEKELGKQGVRGAVSTLLQAPELPTKTRTSRRAAEHRDTTDVPEMPHSKGQGESDPLDHEPAVSPLLPRKERGPPEGGLNEDERLLPKDKKTNLFSALIKKKKKTAPTPPKRSSSFREMDGQPERRGAGEEEGRDISNGALAFTPLDTADPAKSPKPSNGAGVPNGALRESGGSGFRSPHLWKKSSTLTSSRLATGEEEGGGSSSKRFLRSCSASCVPHGAKDTEWRSVTLPRDLQSTGRQFDSSTFGGHKSEKPALPRKRAGENRSDQVTRGTVTPPPRLVKKNEEAADEVFKDIMESSPGSSPPNLTPKPLRRQVTVAPASGLPHKEEAGKGSALGTPAAAEPVTPTSKAGSGAPGGTSKGPAEESRVRRHKHSSESPGRDKGKLSRLKPAPPPPPAASAGKAGGKPSQSPSQEAAGEAVLGAKTKATSLVDAVNSDAAKPSQPGEGLKKPVLPATPKPQSAKPSGTPISPAPVPSTLPSASSAL

In [14]:
!wc -l /content/proteins.fasta


808 /content/proteins.fasta


In [15]:
!mkdir -p /content/mmseqs
!mmseqs createdb /content/proteins.fasta /content/mmseqs/db

createdb /content/proteins.fasta /content/mmseqs/db 

MMseqs Version:       	15-6f452+ds-2
Database type         	0
Shuffle input database	true
Createdb mode         	0
Write lookup file     	1
Offset of numeric ids 	0
Compressed            	0
Verbosity             	3

Converting sequences
[304] 0s 3ms
Time for merging to db_h: 0h 0m 0s 2ms
Time for merging to db: 0h 0m 0s 2ms
Database type: Aminoacid
Time for processing: 0h 0m 0s 11ms


In [16]:
!mmseqs cluster /content/mmseqs/db /content/mmseqs/clusters /content/mmseqs/tmp --min-seq-id 0.4 -c 0.8 --cov-mode 0

Create directory /content/mmseqs/tmp
cluster /content/mmseqs/db /content/mmseqs/clusters /content/mmseqs/tmp --min-seq-id 0.4 -c 0.8 --cov-mode 0 

MMseqs Version:                     	15-6f452+ds-2
Substitution matrix                 	aa:blosum62.out,nucl:nucleotide.out
Seed substitution matrix            	aa:VTML80.out,nucl:nucleotide.out
Sensitivity                         	4
k-mer length                        	0
Target search mode                  	0
k-score                             	seq:2147483647,prof:2147483647
Alphabet size                       	aa:21,nucl:5
Max sequence length                 	65535
Max results per query               	20
Split database                      	0
Split mode                          	2
Split memory limit                  	0
Coverage threshold                  	0.8
Coverage mode                       	0
Compositional bias                  	1
Compositional bias                  	1
Diagonal scoring                    	true
Exact k-mer matching  

In [17]:
!mmseqs createtsv /content/mmseqs/db /content/mmseqs/db /content/mmseqs/clusters /content/mmseqs/clusters.tsv

createtsv /content/mmseqs/db /content/mmseqs/db /content/mmseqs/clusters /content/mmseqs/clusters.tsv 

MMseqs Version:                 	15-6f452+ds-2
First sequence as representative	false
Target column                   	1
Add full header                 	false
Sequence source                 	0
Database output                 	false
Threads                         	2
Compressed                      	0
Verbosity                       	3

Time for merging to clusters.tsv: 0h 0m 0s 0ms
Time for processing: 0h 0m 0s 2ms


In [18]:
!head /content/mmseqs/clusters.tsv
!wc -l /content/mmseqs/clusters.tsv

P46527	P46527
P20936	P20936
Q9Y6E7	Q9Y6E7
Q9NQC7	Q9NQC7
P12270	P12270
Q9H211	Q9H211
P62324	P62324
Q9NZN5	Q9NZN5
Q06455	Q06455
P49815	P49815
404 /content/mmseqs/clusters.tsv


In [19]:
clusters_df = pd.read_csv('/content/mmseqs/clusters.tsv', sep='\t', header=None,
                          names=['cluster_representative', 'accession'])
print(f"Shape : {clusters_df.shape}")
print(clusters_df.head())

# Mapping accession → cluster_representative
accession_to_cluster = dict(zip(clusters_df['accession'], clusters_df['cluster_representative']))

# Sanity checks
assert len(accession_to_cluster) == 404, "Doublon d'accession détecté"
missing = set(accessions_404) - set(accession_to_cluster.keys())
assert not missing, f"Accessions manquantes : {missing}"

n_clusters = len(set(accession_to_cluster.values()))
print(f"Nombre de clusters uniques : {n_clusters}")

Shape : (404, 2)
  cluster_representative accession
0                 P46527    P46527
1                 P20936    P20936
2                 Q9Y6E7    Q9Y6E7
3                 Q9NQC7    Q9NQC7
4                 P12270    P12270
Nombre de clusters uniques : 363


In [20]:
from collections import Counter

accession_to_cluster = {}
with open('/content/mmseqs/clusters.tsv', 'r') as f:
    for line in f:
        cluster_head, accession = line.strip().split('\t')
        accession_to_cluster[accession] = cluster_head

cluster_sizes = Counter(accession_to_cluster.values())
sizes_distribution = Counter(cluster_sizes.values())

print(f"Nombre de clusters : {len(cluster_sizes)}")
print(f"Distribution des tailles :")
for size, count in sorted(sizes_distribution.items()):
    print(f"  {count} cluster(s) de taille {size}")

# Les plus gros clusters (les vraies familles paralogs)
top_clusters = sorted(cluster_sizes.items(), key=lambda x: -x[1])[:10]
print(f"\nTop 10 des plus gros clusters :")
for cluster, size in top_clusters:
    if size > 1:
        members = [acc for acc, cl in accession_to_cluster.items() if cl == cluster]
        print(f"  Cluster {cluster} ({size} membres) : {members}")

Nombre de clusters : 363
Distribution des tailles :
  333 cluster(s) de taille 1
  26 cluster(s) de taille 2
  2 cluster(s) de taille 3
  1 cluster(s) de taille 5
  1 cluster(s) de taille 8

Top 10 des plus gros clusters :
  Cluster P06239 (8 membres) : ['P06239', 'P08631', 'P07948', 'P42685', 'P06241', 'P07947', 'P09769', 'P12931']
  Cluster P62834 (5 membres) : ['P62834', 'P01116', 'P01111', 'P01112', 'P62070']
  Cluster Q9P1W9 (3 membres) : ['Q9P1W9', 'P11309', 'Q86V86']
  Cluster Q9NZH4 (3 membres) : ['Q9NZH4', 'Q9NZH5', 'O95997']
  Cluster P23760 (2 membres) : ['P23760', 'P23759']
  Cluster Q9UHB7 (2 membres) : ['Q9UHB7', 'P51825']
  Cluster P25800 (2 membres) : ['P25800', 'P25791']
  Cluster Q12778 (2 membres) : ['Q12778', 'O43524']
  Cluster P16234 (2 membres) : ['P16234', 'P09619']
  Cluster P42772 (2 membres) : ['P42772', 'P42771']


In [21]:
from sklearn.model_selection import GroupKFold

n_splits = 5
gkf = GroupKFold(n_splits=n_splits)

In [22]:
accessions = features_ref['accession'].tolist()
labels = features_ref['label'].tolist()
groups = [accession_to_cluster[acc] for acc in accessions]


X_placeholder = np.arange(len(accessions))
y = np.array(labels)
groups = np.array(groups)

assert len(X_placeholder) == len(y) == len(groups) == 404

In [23]:
folds = list(gkf.split(X_placeholder, y, groups))
print(f"Nombre de folds : {len(folds)}")
for i, (train_idx, test_idx) in enumerate(folds):
    print(f"  Fold {i} : train={len(train_idx)}, test={len(test_idx)}")

Nombre de folds : 5
  Fold 0 : train=323, test=81
  Fold 1 : train=323, test=81
  Fold 2 : train=323, test=81
  Fold 3 : train=323, test=81
  Fold 4 : train=324, test=80


In [24]:
all_test_indices = []
for train_idx, test_idx in folds:
    all_test_indices.extend(test_idx.tolist())

assert len(all_test_indices) == 404, f"Attendu 404 indices de test au total, trouvé {len(all_test_indices)}"
assert len(set(all_test_indices)) == 404, "Doublon détecté : une protéine apparaît dans plusieurs folds de test"
print("Invariant 1 OK : partition parfaite (chaque protéine dans exactement 1 fold de test)")

Invariant 1 OK : partition parfaite (chaque protéine dans exactement 1 fold de test)


In [25]:
from collections import defaultdict
cluster_to_folds = defaultdict(set)
for fold_id, (train_idx, test_idx) in enumerate(folds):
    for i in test_idx:
        cluster = groups[i]
        cluster_to_folds[cluster].add(fold_id)

split_clusters = {c: fs for c, fs in cluster_to_folds.items() if len(fs) > 1}

In [31]:
print(y)

['oncogene' 'oncogene' 'oncogene' 'oncogene' 'oncogene' 'oncogene'
 'oncogene' 'oncogene' 'oncogene' 'oncogene' 'oncogene' 'oncogene'
 'oncogene' 'oncogene' 'oncogene' 'oncogene' 'oncogene' 'oncogene'
 'oncogene' 'oncogene' 'oncogene' 'oncogene' 'oncogene' 'oncogene'
 'oncogene' 'oncogene' 'oncogene' 'oncogene' 'oncogene' 'oncogene'
 'oncogene' 'oncogene' 'oncogene' 'oncogene' 'oncogene' 'oncogene'
 'oncogene' 'oncogene' 'oncogene' 'oncogene' 'oncogene' 'oncogene'
 'oncogene' 'oncogene' 'oncogene' 'oncogene' 'oncogene' 'oncogene'
 'oncogene' 'oncogene' 'oncogene' 'oncogene' 'oncogene' 'oncogene'
 'oncogene' 'oncogene' 'oncogene' 'oncogene' 'oncogene' 'oncogene'
 'oncogene' 'oncogene' 'oncogene' 'oncogene' 'oncogene' 'oncogene'
 'oncogene' 'oncogene' 'oncogene' 'oncogene' 'oncogene' 'oncogene'
 'oncogene' 'oncogene' 'oncogene' 'oncogene' 'oncogene' 'oncogene'
 'oncogene' 'oncogene' 'oncogene' 'oncogene' 'oncogene' 'oncogene'
 'oncogene' 'oncogene' 'oncogene' 'oncogene' 'oncogene' 'oncog

In [39]:
from collections import Counter

print("Distribution des labels par fold (test set) :")
print(f"Ratio global : 55.7% oncogène / 44.3% tumor_suppressor\n")

for fold_id, (train_idx, test_idx) in enumerate(folds):
    labels_in_test = [y[i] for i in test_idx]
    counts = Counter(labels_in_test)
    total = len(test_idx)

    n_onco = counts.get('oncogene', 0)
    n_sup = counts.get('tumor_suppressor', 0)

    print(f"  Fold {fold_id} : {n_onco} onco ({100*n_onco/total:.1f}%) / "
          f"{n_sup} sup ({100*n_sup/total:.1f}%)")

Distribution des labels par fold (test set) :
Ratio global : 55.7% oncogène / 44.3% tumor_suppressor

  Fold 0 : 43 onco (53.1%) / 38 sup (46.9%)
  Fold 1 : 47 onco (58.0%) / 34 sup (42.0%)
  Fold 2 : 45 onco (55.6%) / 36 sup (44.4%)
  Fold 3 : 50 onco (61.7%) / 31 sup (38.3%)
  Fold 4 : 40 onco (50.0%) / 40 sup (50.0%)


In [53]:
from sklearn.base import clone
from sklearn.metrics import f1_score, balanced_accuracy_score, matthews_corrcoef, roc_auc_score
from sklearn.preprocessing import LabelEncoder
import numpy as np


def evaluate_model(model_factory, X, y, groups, folds, seeds):
    """
    Évalue un modèle sur un split donné avec plusieurs seeds.

    Args:
        model_factory: fonction qui prend une seed et retourne un modèle scikit-learn frais
        X: matrice de features, shape (n_samples, n_features)
        y: labels 1D, shape (n_samples,) — strings ou entiers
        groups: identifiants de groupe (clusters), shape (n_samples,)
        folds: liste de tuples (train_idx, test_idx) — déjà calculés par GroupKFold
        seeds: liste d'entiers, une seed par run

    Returns:
        dict avec les scores individuels et leurs statistiques
    """
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    positive_class_idx = 1
        # --- Structure de stockage des résultats ---
    metric_names = ['f1_macro', 'balanced_accuracy', 'mcc', 'roc_auc']
    results = {name: {'scores': [], 'mean': None, 'std': None} for name in metric_names}

    for seed in seeds:
      for train_idx, test_idx in folds:
          model = model_factory(seed)
          X_train, X_test = X[train_idx], X[test_idx]
          y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]
          model.fit(X_train, y_train)
          y_pred = model.predict(X_test)
          y_proba = model.predict_proba(X_test)[:, positive_class_idx]
          results['f1_macro']['scores'].append(
              f1_score(y_test, y_pred, average='macro')
          )
          results['balanced_accuracy']['scores'].append(
              balanced_accuracy_score(y_test, y_pred)
          )
          results['mcc']['scores'].append(
              matthews_corrcoef(y_test, y_pred)
          )
          results['roc_auc']['scores'].append(
              roc_auc_score(y_test, y_proba)
          )

              # --- Calcul des statistiques finales ---
    for name in metric_names:
      scores = results[name]['scores']
      results[name]['mean'] = float(np.mean(scores))
      results[name]['std'] = float(np.std(scores))

    return results

In [54]:
from sklearn.ensemble import RandomForestClassifier

DROP_COLS = ['accession', 'gene', 'label']
feature_columns = [c for c in features_ref.columns if c not in DROP_COLS]
X_baseline = features_ref[feature_columns].values

print(f"X shape : {X_baseline.shape}")
print(f"Features : {feature_columns[:5]}... ({len(feature_columns)} au total)")

def rf_factory(seed):
    return RandomForestClassifier(
        n_estimators=300,
        max_depth=5,
        class_weight='balanced',
        random_state=seed,
        n_jobs=-1,
    )

seeds = [42, 43, 44, 45, 46]

print("\nÉvaluation en cours (5 seeds × 5 folds = 25 runs)...")
results_baseline = evaluate_model(
    model_factory=rf_factory,
    X=X_baseline,
    y=y,
    groups=groups,
    folds=folds,
    seeds=seeds,
)

# --- Afficher les résultats ---
print("\nRésultats OncoLake baseline (RF sur 25 features handcraftées) :")
print(f"  F1 macro          : {results_baseline['f1_macro']['mean']:.4f} ± {results_baseline['f1_macro']['std']:.4f}")
print(f"  Balanced accuracy : {results_baseline['balanced_accuracy']['mean']:.4f} ± {results_baseline['balanced_accuracy']['std']:.4f}")
print(f"  MCC               : {results_baseline['mcc']['mean']:.4f} ± {results_baseline['mcc']['std']:.4f}")
print(f"  AUC-ROC           : {results_baseline['roc_auc']['mean']:.4f} ± {results_baseline['roc_auc']['std']:.4f}")

X shape : (404, 25)
Features : ['n_residues_structure', 'plddt_mean', 'pct_low_confidence', 'radius_of_gyration', 'aa_A']... (25 au total)

Évaluation en cours (5 seeds × 5 folds = 25 runs)...

Résultats OncoLake baseline (RF sur 25 features handcraftées) :
  F1 macro          : 0.4741 ± 0.0422
  Balanced accuracy : 0.4791 ± 0.0409
  MCC               : -0.0427 ± 0.0813
  AUC-ROC           : 0.4819 ± 0.0378


In [55]:
import json

results_path = '/content/drive/MyDrive/oncolake_torchprotein/results'
!mkdir -p "{results_path}"

# On sauvegarde en JSON pour lisibilité et portabilité
output_file = f"{results_path}/baseline_oncolake_handcrafted.json"
with open(output_file, 'w') as f:
    json.dump(results_baseline, f, indent=2)

print(f"Résultats sauvegardés dans : {output_file}")

Résultats sauvegardés dans : /content/drive/MyDrive/oncolake_torchprotein/results/baseline_oncolake_handcrafted.json


This notebook establishes the evaluation protocol shared by all three models
of the project, and reproduces the OncoLake handcrafted baseline with a
methodologically corrected pipeline.
The handcrafted feature baseline performs at random chance under the corrected
protocol, confirming that global structural descriptors do not carry the
discriminative signal for oncogene vs tumor suppressor classification. The
next notebook tests whether learned geometric embeddings (TorchProtein GNN)
can recover a signal invisible to these features.

Saved artifacts:
- `results/baseline_oncolake_handcrafted.json` (25 individual scores + stats)